## 91. How would you secure a **production GenAI application**?

> **Interview Answer:** “I use a **defense-in-depth security architecture** covering identity, data, prompts, model access, tools, APIs, infrastructure, and monitoring. I apply least privilege and never rely on the LLM itself for authorization.”

### Security architecture

```text
User
 ↓
API Gateway / WAF
 ↓
Authentication + Authorization
 ↓
Input Validation
 ↓
RAG / Agent Layer
 ↓
LLM
 ↓
Output Validation
 ↓
Tool Authorization
 ↓
Enterprise APIs
```

### Key controls

**1. Identity & Access**
- Entra ID / IAM
- OAuth2 / JWT
- RBAC / ABAC
- Least privilege
- Managed Identity

**2. Data Security**
- Encryption in transit + at rest
- Key Vault / Secrets Manager
- PII masking/redaction
- Tenant isolation
- Document-level access control

**3. LLM Security**
- Prompt-injection protection
- Input/output validation
- Guardrails
- Sensitive-data filtering
- Don't expose system prompts or secrets

**4. Agent / Tool Security**

```text
LLM selects tool
      ↓
Authorization Layer
      ↓
Is tool allowed?
 ├── Yes → Execute
 └── No  → Block
```

- Tool allowlisting
- Per-tool permissions
- API scopes
- Human approval for high-risk actions
- Rate limits

**5. RAG Security**
- Tenant-aware metadata filtering
- Document ACL filtering
- Prevent cross-tenant retrieval
- Validate retrieved content
- Treat retrieved documents as **untrusted data**

**6. API / Infrastructure**
- API Gateway + WAF
- Rate limiting
- Network isolation/private endpoints
- Container security
- Dependency scanning
- Vulnerability management

**7. Monitoring & Audit**
- Authentication events
- Prompt-injection attempts
- Tool calls
- Data-access events
- LLM requests/responses where appropriate
- Security alerts
- Audit trails

### Production flow

```text
Authentication
      ↓
Authorization
      ↓
Input Guardrails
      ↓
RAG / Agent
      ↓
LLM
      ↓
Output Guardrails
      ↓
Tool Authorization
      ↓
Enterprise API
      ↓
Audit + Monitoring
```

**Interview one-liner:**

> **“For production GenAI, I apply defense-in-depth: strong IAM and least privilege, encrypted and tenant-isolated data, prompt-injection defenses, strict tool authorization, RAG access filtering, API security, output validation, and end-to-end audit and monitoring.”**

## 92. How do you implement **Microsoft Entra ID**?

> **Interview Answer:** “I use Microsoft Entra ID for centralized authentication and authorization. I register the application, configure OAuth 2.0/OpenID Connect, obtain an access token, and validate the JWT in my API. I then use claims such as user ID, roles, scopes, and tenant ID for authorization.”

### Production flow

```text
User
 ↓
Microsoft Entra ID
 ↓
Login / OAuth2
 ↓
Access Token (JWT)
 ↓
FastAPI / API Gateway
 ↓
Token Validation
 ↓
Claims / Roles / Scopes
 ↓
Authorization
 ↓
LLM / RAG / Tools
```

### Implementation steps

**1. App Registration**
- Register application in Entra ID
- Configure redirect URI if required
- Create required API scopes/roles

**2. Authentication**

```text
User → Entra ID → Access Token
```

Use **OAuth 2.0 / OpenID Connect**.

**3. API validates JWT**

Validate:

```text
Issuer
Audience
Signature
Expiration
Scopes / Roles
```

**4. Authorization**

```text
Token Claims
    ↓
Role / Scope Check
    ↓
Allowed?
 ├── Yes → API
 └── No  → 403
```

Example:

```text
Role: HR_MANAGER
Scope: shift.write
Tenant: T001
```

Only then:

```text
create_shift()
```

### For GenAI / RAG

I would propagate identity into the retrieval layer:

```text
Entra User
   ↓
Tenant / User / Roles
   ↓
Azure AI Search Filter
   ↓
Only Authorized Documents
   ↓
LLM
```

This prevents **cross-tenant or unauthorized document retrieval**.

### Service-to-service

For backend services, I prefer:

- **Managed Identity**
- Entra ID tokens
- RBAC
- No hardcoded credentials

```text
FastAPI
   ↓
Managed Identity
   ↓
Entra ID Token
   ↓
Azure OpenAI / Azure AI Search
```

**Interview one-liner:**

> **“I implement Entra ID using OAuth2/OIDC for authentication, JWT validation and RBAC/scopes for authorization, and Managed Identity for service-to-service access. In RAG, I propagate the user's identity and authorization claims into the search layer to enforce document-level access.”**

## 93. Managed Identity vs **API Key**

> **Interview Answer:** “Managed Identity is my preferred approach for Azure production workloads because it eliminates storing credentials in the application. API keys are simpler but require secure storage, rotation, and lifecycle management.”

| | **Managed Identity** | **API Key** |
|---|---|---|
| Credential storage | No application secret | Secret required |
| Authentication | Entra ID | Static key |
| Rotation | Azure-managed | Application/process managed |
| Security | Higher | Lower |
| Best use | Production Azure services | Development / systems that require keys |

### Production flow

```text
FastAPI / Azure App
       ↓
Managed Identity
       ↓
Microsoft Entra ID
       ↓
Access Token
       ↓
Azure OpenAI / AI Search
```

**One-liner:**

> **“For Azure-to-Azure communication, I prefer Managed Identity; I use API keys only when the target service requires them or when Managed Identity isn't available.”**

---

## 94. How would you use **Azure Key Vault**?

> **Interview Answer:** “I use Azure Key Vault as the centralized secret-management layer for credentials, API keys, certificates, and other sensitive configuration. The application accesses Key Vault using Managed Identity rather than storing secrets in source code or configuration files.”

### Architecture

```text
FastAPI / Container
       ↓
Managed Identity
       ↓
Microsoft Entra ID
       ↓
Azure Key Vault
       ↓
Secret / Key / Certificate
       ↓
Application
```

### What I store

- LLM API keys
- Third-party API credentials
- Database connection secrets
- Certificates
- Encryption keys

### Security controls

- RBAC
- Managed Identity
- Secret rotation
- Access policies
- Audit logging
- No secrets in Git
- No secrets in Docker images

**One-liner:**

> **“Key Vault centralizes secret management, while Managed Identity provides secure, passwordless access to those secrets.”**

---

## 95. How do you protect **LLM API credentials**?

> **Interview Answer:** “I never hardcode LLM credentials in source code, Git, notebooks, or Docker images. In Azure production, I prefer Managed Identity where supported; otherwise I store the API key in Key Vault and retrieve it securely at runtime.”

### Production architecture

```text
              Azure Production
                     │
              ┌──────┴──────┐
              ↓             ↓
        Managed Identity   Key Vault
              ↓             ↓
              └──────┬──────┘
                     ↓
              LLM API Client
                     ↓
                Azure OpenAI
```

### Controls

- **Managed Identity** where supported
- **Azure Key Vault** for required secrets
- Environment variables only for non-sensitive configuration
- No credentials in source control
- Secret rotation
- RBAC / least privilege
- Private endpoints/network restrictions
- Audit access
- Credential scanning in CI/CD

### Interview one-liner

> **“My preferred pattern is Managed Identity for passwordless authentication; when an API key is unavoidable, I store it in Key Vault, grant the application least-privilege access, rotate it periodically, and never expose it in code, logs, repositories, or client-side applications.”**

## 96. How do you implement **RBAC**?

> **Interview Answer:** “I implement RBAC using **Microsoft Entra ID roles/groups** and enforce authorization at the API and tool layers. Authentication identifies the user, while roles and scopes determine what operations they are allowed to perform.”

```text
User
 ↓
Entra ID
 ↓
JWT Token
 ↓
Roles / Scopes
 ↓
Authorization Middleware
 ↓
Allowed Resource / Tool
```

### Example

```text
HR_VIEWER  → Read shifts
HR_MANAGER → Read + Create + Update
HR_ADMIN   → Full access
```

For an agent:

```text
LLM → create_shift
          ↓
Authorization Check
          ↓
User has shift.write?
      ┌───┴───┐
     Yes      No
      ↓        ↓
   Execute    Block
```

**One-liner:**

> **“I enforce RBAC outside the LLM using Entra ID roles/scopes and authorization middleware, with least-privilege access to APIs, tools and data.”**

---

## 97. How do you **isolate tenant data**?

> **Interview Answer:** “I use tenant isolation at multiple layers—**identity, application, database, and vector search**. Every request carries a validated `tenant_id`, and all data access is scoped to that tenant.”

### Architecture

```text
User
 ↓
Entra ID
 ↓
Tenant ID
 ↓
API Authorization
 ↓
Tenant Filter
 ├── Database
 ├── Blob Storage
 └── Vector/Search Index
```

### Metadata example

```python
{
    "document_id": "DOC101",
    "tenant_id": "TENANT_A",
    "department": "HR"
}
```

Query:

```text
tenant_id = TENANT_A
```

### Stronger isolation options

- **Logical isolation** → `tenant_id` on every record/chunk
- **Database-level isolation** → Row-Level Security
- **Physical isolation** → Separate database/index/storage per tenant
- Separate encryption keys where required
- Tenant-aware authorization

**One-liner:**

> **“For most SaaS RAG systems I use tenant-aware identity and metadata filtering, with database-level or physically separate storage for high-isolation requirements.”**

---

## 98. How do you prevent **cross-tenant retrieval**?

> **Interview Answer:** “I treat tenant isolation as a security boundary, not just a search filter. I extract and validate the tenant identity from the authenticated token, then enforce that tenant ID in the retrieval query. The LLM is never allowed to decide which tenant's data it can access.”

### Secure RAG flow

```text
User
 ↓
Entra ID
 ↓
JWT
 ↓
Validated tenant_id
 ↓
Authorization Layer
 ↓
Azure AI Search
 ↓
tenant_id = authenticated_tenant
 ↓
Hybrid Search
 ↓
Reranking
 ↓
LLM
```

### Example

User belongs to:

```text
tenant_id = TENANT_A
```

Search must effectively be:

```text
tenant_id == TENANT_A
AND
<user query>
```

Not:

```text
<user query>        ❌
```

### Defense-in-depth

- Tenant ID from **trusted identity**, not user input
- Mandatory server-side metadata filter
- API authorization
- Database Row-Level Security where applicable
- Document ACLs
- Never allow LLM to construct authorization filters
- Audit retrieval events
- Test cross-tenant attack scenarios

### Critical interview point

> **“The tenant ID must come from the authenticated identity and be enforced server-side. I never trust a tenant ID supplied in the user's prompt, and I don't rely on the LLM to enforce authorization.”**

**Interview one-liner:**

> **“I prevent cross-tenant retrieval by deriving tenant identity from Entra ID, enforcing tenant-level authorization before retrieval, applying mandatory server-side filters in the search layer, and validating the complete path with security tests.”**

## 99. How do you handle **PII**?

> **Answer:** “I identify PII at ingestion and runtime, then apply **classification, masking/redaction, encryption, access control, and audit logging**. I minimize the PII sent to the LLM and retain only what is business-required.”

```text
Input / Document
      ↓
PII Detection
      ↓
Classify
      ↓
Mask / Redact
      ↓
LLM
      ↓
Output PII Check
      ↓
Response
```

- PII detection → names, email, phone, Aadhaar, etc.
- Data minimization
- Mask/redact before LLM
- Encryption at rest/in transit
- RBAC + tenant isolation
- Audit access
- Retention/deletion policies

**One-liner:**

> **“My principle is to detect and minimize PII before it reaches the LLM, enforce access controls, and scan the output as well.”**

---

## 100. How do you implement **data masking**?

> **Answer:** “I detect sensitive fields and replace them with tokens or masked values before sending data to the LLM.”

```python
text = "Contact john@example.com, phone 9876543210"

masked = "Contact [EMAIL], phone [PHONE]"
```

### Flow

```text
Raw Data
   ↓
PII Detector
   ↓
Mask / Tokenize
   ↓
LLM
   ↓
Detokenize if required
   ↓
Authorized Response
```

Example:

```text
9876543210
     ↓
98XXXX3210
```

For reversible workflows, I prefer **tokenization** rather than storing the original value in the prompt.

**One-liner:**

> **“I mask or tokenize sensitive fields before LLM processing and only detokenize downstream when there is a legitimate business requirement and authorization.”**

---

## 101. How do you prevent sensitive information from reaching the **LLM**?

> **Answer:** “I put a **data-loss-prevention layer before the LLM**. It inspects user input, retrieved documents, and tool outputs, removes or masks sensitive information, and only then constructs the final prompt.”

```text
User Input
    +
RAG Context
    +
Tool Output
    ↓
Sensitive Data Detection
    ↓
Redaction / Masking
    ↓
Policy Check
    ↓
LLM
```

### Controls

- PII detection
- Data classification
- Redaction/masking
- Metadata-based access control
- Tenant isolation
- Document ACLs
- Prompt minimization
- Output scanning
- Logging without sensitive payloads

**Critical point:**

> **“I don't depend on the LLM to protect sensitive data; the application security layer enforces the policy before the request reaches the model.”**

---

## 102. How do you implement **AI Content Safety**?

> **Answer:** “I implement content safety at both **input and output boundaries**. I classify user prompts and generated responses against policies and block, redact, or escalate content that violates those policies.”

```text
User Input
   ↓
Content Safety
   ↓
LLM
   ↓
Output
   ↓
Content Safety
   ↓
Response
```

### Controls

- Harmful-content detection
- Prompt-injection detection
- PII detection
- Profanity/toxicity filtering
- Unsafe content blocking
- Output validation
- Rate limiting
- Human escalation

For Azure workloads, I can use **Azure AI Content Safety** as a dedicated safety service, combined with application-level policies.

**One-liner:**

> **“I use content-safety checks on both input and output, with policy-based blocking or escalation, rather than assuming the model will always generate safe content.”**

---

## 103. Azure AI Content Safety vs **application-level guardrails**?

> **Answer:** “Azure AI Content Safety provides specialized detection capabilities, while application-level guardrails enforce my **business-specific security and operational policies**. I use both as complementary layers.”

| Azure AI Content Safety | Application Guardrails |
|---|---|
| Content classification | Business rules |
| Harmful content detection | Authorization |
| Safety categories | Tool permissions |
| PII-related safety capabilities | Schema validation |
| Model-independent safety layer | Tenant validation |
| Safety-focused | Application-specific |

### Architecture

```text
             Input
               ↓
    Application Guardrails
               ↓
      Azure AI Content Safety
               ↓
              LLM
               ↓
      Azure AI Content Safety
               ↓
    Application Guardrails
               ↓
            Output
```

**One-liner:**

> **“Content Safety answers ‘is this content unsafe?’, while application guardrails answer ‘is this action allowed in my business context?’”**

---

## 104. How do you defend against **malicious documents in RAG**?

> **Answer:** “I treat every external document as **untrusted input**. A document can contain prompt-injection instructions, malicious content, or misleading information, so I validate it before indexing and never treat retrieved text as system instructions.”

### Secure RAG

```text
Document
   ↓
Malware / File Validation
   ↓
Content Extraction
   ↓
PII / Safety / Injection Detection
   ↓
Sanitize
   ↓
Chunk + Embed
   ↓
Index
```

At retrieval:

```text
Retrieved Document
       ↓
Treat as DATA
       ↓
Do NOT execute instructions
       ↓
LLM
```

### Controls

- File-type validation
- Malware scanning
- Content sanitization
- Prompt-injection detection
- Source trust/allowlisting
- Metadata/ACL validation
- Tenant filtering
- Citation/source tracking
- Never execute instructions found in documents

**Critical interview point:**

> **“Retrieved content has the same trust level as user-provided data; it must never be allowed to override system instructions or directly invoke tools.”**

---

## 105. How do you **audit agent tool calls**?

> **Answer:** “I create an audit record for every tool invocation containing the **user identity, tenant, agent, tool name, timestamp, request ID, parameters, authorization decision, result status, latency, and error information**. Sensitive values are redacted.”

```text
User Request
     ↓
Agent
     ↓
Tool Authorization
     ↓
Tool Call
     ↓
Audit Event
     ↓
Enterprise API
     ↓
Result
     ↓
Audit Event
```

### Example audit record

```python
{
    "request_id": "REQ123",
    "user_id": "USER456",
    "tenant_id": "TENANT_A",
    "agent": "ShiftAgent",
    "tool": "create_shift",
    "authorization": "allowed",
    "status": "success",
    "latency_ms": 240
}
```

### What I track

- Who initiated the action
- Which agent made the decision
- Which tool was selected
- Tool arguments — **redacted where sensitive**
- Authorization result
- API response/status
- Timestamp
- Latency
- Retry count
- Failure reason
- Human approval, if applicable

**One-liner:**

> **“Every agent tool call is treated as an auditable business action: I log identity, tenant, tool, authorization, outcome and trace ID, while redacting sensitive payloads and maintaining immutable audit records.”**